In [3]:
import os
import pickle
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from skimage.io import imread
from skimage.transform import resize
from PIL import Image, ImageFile
from skimage.color import gray2rgb, rgba2rgb
import numpy as np
from skimage.feature import hog
from skimage.transform import resize
from skimage.color import rgb2gray

In [4]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

input_dir = './images'
categories = ['alopecia', 'no_alopecia', "receding_hairline"]

valid_ext = ('.jpg', '.jpeg', '.png')

data = []
labels = []

for category_idx, category in enumerate(categories):
    folder_path = os.path.join(input_dir, category)

    for file in os.listdir(folder_path):
        if not file.lower().endswith(valid_ext):
            continue

        img_path = os.path.join(folder_path, file)

        try:
            # load image
            img = Image.open(img_path).convert("RGB")
            img = np.array(img)

            # resize
            img = resize(img, (128, 128), anti_aliasing=True)

            # convert to grayscale
            img_gray = rgb2gray(img)

            # extract HOG features
            features = hog(
                img_gray,
                pixels_per_cell=(16, 16),
                cells_per_block=(2, 2),
                feature_vector=True
            )

            data.append(features)
            labels.append(category_idx)

        except Exception as e:
            print(f"Skipping {img_path}: {e}")

data = np.asarray(data)
labels = np.asarray(labels)

Skipping ./images\alopecia\98079763-0-image-m-13_1746523682040.jpg: Failed to decode frame 0: No codec available


In [5]:
print(data.shape)
print(labels.shape)

print(features.shape)

(2356, 1764)
(2356,)
(1764,)


## Support Vector Multi-Classification

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# =========================
# SPLIT
# =========================
x_train, x_test, y_train, y_test = train_test_split(
    data,
    labels,
    test_size=0.2,
    shuffle=True,
    stratify=labels,
    random_state=42
)

# =========================
# PIPELINE
# =========================
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(
        C=10,                 
        kernel="rbf",       
        gamma="scale",
        probability=True,   
        class_weight="balanced",
        random_state=42
    ))
])

# =========================
# TRAIN
# =========================
model.fit(x_train, y_train)

# =========================
# PREDICT
# =========================
y_pred = model.predict(x_test)

# =========================
# METRICS (MULTICLASS)
# =========================
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"{accuracy*100:.2f}% accuracy")
print(f"{precision*100:.2f}% precision")
print(f"{recall*100:.2f}% recall")
print(f"{f1*100:.2f}% f1-score")

# =========================
# SAVE MODEL
# =========================
pickle.dump(model, open("./model.p", "wb"))

82.63% accuracy
79.19% precision
66.89% recall
70.65% f1-score


## Gaussian Naive Bayes

In [5]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pickle

# split
x_train, x_test, y_train, y_test = train_test_split(
    data,
    labels,
    test_size=0.2,
    shuffle=True,
    stratify=labels,
    random_state=42
)

# modelo direto (sem grid search)
model = XGBClassifier(
    objective='multi:softprob',
    eval_metric='mlogloss',
    num_class=len(set(y_train)),
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

# treino
model.fit(x_train, y_train)

# predição
y_prediction = model.predict(x_test)

# métricas (multiclass)
score = accuracy_score(y_test, y_prediction)
precision = precision_score(y_test, y_prediction, average='macro')
recall = recall_score(y_test, y_prediction, average='macro')
f1 = f1_score(y_test, y_pred, average="macro")

print(f'{score*100:.2f}% accuracy')
print(f'{precision*100:.2f}% precision')
print(f'{recall*100:.2f}% recall')
print(f"{f1*100:.2f}% f1-score")

# salvar modelo
#pickle.dump(model, open('./model.p', 'wb'))

80.13% accuracy
82.98% precision
62.24% recall
69.71% f1-score


## Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
import pickle

# =========================
# SPLIT
# =========================
x_train, x_test, y_train, y_test = train_test_split(
    data,
    labels,
    test_size=0.2,
    shuffle=True,
    stratify=labels,
    random_state=42
)

# =========================
# MODEL
# =========================
model = RandomForestClassifier(random_state=42)

# =========================
# HYPERPARAMETERS
# =========================
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

# =========================
# GRID SEARCH
# =========================
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(x_train, y_train)

# =========================
# BEST MODEL
# =========================
best_estimator = grid_search.best_estimator_

# =========================
# PREDICTION
# =========================
y_prediction = best_estimator.predict(x_test)

# =========================
# METRICS (MULTICLASS SAFE)
# =========================
score = accuracy_score(y_test, y_prediction)
precision = precision_score(y_test, y_prediction, average='macro', zero_division=0)
recall = recall_score(y_test, y_prediction, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro")

print(f'{score*100:.2f}% accuracy')
print(f'{precision*100:.2f}% precision')
print(f'{recall*100:.2f}% recall')
print(f"{f1*100:.2f}% f1-score")

# =========================
# SAVE MODEL
# =========================
#pickle.dump(best_estimator, open('./model.p', 'wb'))

Fitting 3 folds for each of 108 candidates, totalling 324 fits


## Extra Trees

In [7]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

# =========================
# 1. TRAIN / TEST SPLIT
# =========================
x_train, x_test, y_train, y_test = train_test_split(
    data,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

# =========================
# 2. MODEL (stronger baseline)
# =========================
model = ExtraTreesClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

# =========================
# 3. PARAM GRID (more robust)
# =========================
param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [None, 20, 40],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=cv,
    n_jobs=-1,
    verbose=2,
    scoring="f1_macro"   # IMPORTANT for imbalanced multiclass
)

# =========================
# 4. TRAIN
# =========================
grid_search.fit(x_train, y_train)

best_model = grid_search.best_estimator_

# =========================
# 5. TEST
# =========================
y_pred = best_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"{accuracy*100:.2f}% accuracy")
print(f"{precision*100:.2f}% precision")
print(f"{recall*100:.2f}% recall")
print(f"{f1*100:.2f}% F1-score (macro)")

# =========================
# 6. SAVE
# =========================
# pickle.dump(best_model, open("./model.p", "wb"))

Fitting 5 folds for each of 108 candidates, totalling 540 fits
79.28% accuracy
75.34% precision
60.82% recall
64.49% F1-score (macro)
